# Experimentos de clustering

Búsqueda sistematica de la mejor segmentación. Se combinan conjuntos de variables
(conducta electoral, trayectoria, PJ 2025, demografia, región), preprocesamientos y
algoritmos con sus hiperparametros. El objetivo es descartar con evidencia las
alternativas debiles y quedarse con pocas candidatas solidas.

Tres etapas:
1. **Búsqueda en malla** sobre una muestra ponderada por lista nominal.
2. **Estabilidad** por bootstrap de las mejores configuraciones.
3. **Afinacion** de las finalistas con todos los datos y comparación contra el modelo
   del notebook 04 (que no se modifica).

## 1. Configuración

In [ ]:
import copy
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.stats import chi2_contingency
from sklearn.cluster import HDBSCAN, Birch, BisectingKMeans, KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import SpectralEmbedding
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

## 2. Parametros y rutas

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DIR_PREPROCESSED = ROOT / "DATA" / "PREPROCESSED"

ELECCIONES = ["P2009", "P2012", "P2015", "P2018", "P2021", "P2024"]
CORTES_EDAD = [17, 24, 29, 39, 49, 59, 69, 79, 200]
ETIQUETAS_EDAD = ["18-24", "25-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]
REGIONES = ["Noroeste", "Noreste", "Occidente", "Centro", "Sur-Sureste"]

K_RANGE = range(3, 9)
K_VALIDO = range(4, 7)  # convocatoria pide 4-6 perfiles
MIN_PCT_PERFIL = 5.0
MAX_PCT_RUIDO = 10.0

N_FIT = 60_000
N_SUB = 8_000
N_EVAL = 10_000
N_SIL = 5_000
N_FINAL = 200_000
N_BOOT = 5
N_CANDIDATOS = 15
N_FINALISTAS = 3
ARI_ESTABLE = 0.8
RANDOM_STATE = 42

## 3. Carga de datos

### 3.1 Celdas con historial

Celdas sección x sexo x grupo de edad generadas en el notebook 03.

In [ ]:
celdas = pd.read_parquet(DIR_PREPROCESSED / "celdas_historial.parquet")
celdas["GRUPO_EDAD"] = celdas["GRUPO_EDAD"].astype(str)
print(f"Celdas: {len(celdas):,}")

### 3.2 Peso poblacional

Mismo criterio que el notebook 04: votantes potenciales 2024 (SV + NV) por celda.

In [ ]:
fed24 = pd.read_parquet(
    DIR_PREPROCESSED / "conteos_federal_2009_2024.parquet",
    columns=["EDOCVE", "SECCIÓN", "SEXO", "EDAD", "AELEC", "SV", "NV"],
    filters=[("AELEC", "==", 2024)],
)
fed24["GRUPO_EDAD"] = pd.cut(fed24["EDAD"], bins=CORTES_EDAD, labels=ETIQUETAS_EDAD)
fed24["VOTANTES"] = fed24["SV"] + fed24["NV"]

pesos = (
    fed24.groupby(["EDOCVE", "SECCIÓN", "SEXO", "GRUPO_EDAD"], observed=True)["VOTANTES"]
    .sum()
    .reset_index()
    .rename(columns={"VOTANTES": "peso"})
)
pesos["GRUPO_EDAD"] = pesos["GRUPO_EDAD"].astype(str)
del fed24

celdas = celdas.merge(pesos, on=["EDOCVE", "SECCIÓN", "SEXO", "GRUPO_EDAD"], how="left")
print(f"Peso total: {celdas['peso'].sum():,.0f} personas")

### 3.3 Participación PJ 2025 y casos completos

Para que todas las configuraciones se comparen sobre la misma población,
se conservan solo las celdas con peso y con tasa PJ 2025.

In [ ]:
secciones = pd.read_parquet(
    DIR_PREPROCESSED / "secciones_historial.parquet", columns=["EDOCVE", "SECCIÓN", "P_PJ2025"]
)
celdas = celdas.merge(secciones, on=["EDOCVE", "SECCIÓN"], how="left")
del secciones

n0, w0 = len(celdas), celdas["peso"].sum()
celdas = celdas.dropna(subset=["peso", "P_PJ2025"]).reset_index(drop=True)
print(f"Celdas completas: {len(celdas):,} de {n0:,} ({celdas['peso'].sum() / w0 * 100:.1f}% del peso)")

### 3.4 Codificacion de variables demográficas

Edad y tipo de sección como ordinales, sexo como indicador de mujer y región como dummies.

In [ ]:
celdas["EDAD_ORD"] = celdas["GRUPO_EDAD"].map({g: i for i, g in enumerate(ETIQUETAS_EDAD)})
celdas["MUJER"] = (celdas["SEXO"] == 1).astype(int)
celdas["TIPOSEC_ORD"] = celdas["TIPOSEC"].astype(str).map({"U": 0, "M": 1, "R": 2})
for r in REGIONES:
    celdas[f"REG_{r}"] = (celdas["REGIÓN"] == r).astype(int)

print(celdas[["EDAD_ORD", "MUJER", "TIPOSEC_ORD"]].isna().sum().to_dict())

## 4. Diseño del experimento

### 4.1 Conjuntos de variables

- **A-D:** solo conducta electoral (indicadores o trayectoria completa, con y sin PJ 2025).
- **E-H:** conducta más demografia (edad, sexo, tipo de sección) y región.
- **I:** solo demografia y región. Es un control: si gana, la conducta no aporta.

In [ ]:
BASE = ["promedio", "brecha_pres", "tendencia"]
INDICADORES = BASE + ["desviación", "P_PJ2025"]
TRAYECTORIA_PJ = ELECCIONES + ["P_PJ2025"]
DEMO = ["EDAD_ORD", "MUJER", "TIPOSEC_ORD"]
REG = [f"REG_{r}" for r in REGIONES]

CONJUNTOS = {
    "A_base": BASE,
    "B_indicadores": INDICADORES,
    "C_trayectoria": ELECCIONES,
    "D_trayectoria_pj": TRAYECTORIA_PJ,
    "E_base_demo": BASE + DEMO,
    "F_indicadores_demo": INDICADORES + DEMO,
    "G_trayectoria_demo": TRAYECTORIA_PJ + DEMO,
    "H_completo": TRAYECTORIA_PJ + DEMO + REG,
    "I_solo_demo": DEMO + REG,
}
for nombre, cols in CONJUNTOS.items():
    print(f"{nombre:20s} {len(cols):2d} variables")

### 4.2 Muestras ponderadas

Las muestras se extraen con probabilidad proporcional al peso, así cada fila representa
aproximadamente el mismo número de personas y las métricas sin ponderar equivalen a
métricas ponderadas. Esto permite usar algoritmos que no aceptan `sample_weight`.

- **eval** (10k): fija, se usa para calcular todas las métricas.
- **fit** (60k): ajuste de KMeans, BisectingKMeans y GMM.
- **sub** (8k, dentro de fit): ajuste de Birch, aglomerativo, espectral y HDBSCAN, que escalan
  mal. Los tres últimos no tienen `predict`; sus etiquetas se extienden con KNN (k=5).

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
p_all = celdas["peso"].to_numpy() / celdas["peso"].sum()
idx_eval = rng.choice(len(celdas), size=N_EVAL, replace=False, p=p_all)

resto = np.setdiff1d(np.arange(len(celdas)), idx_eval)
p_resto = p_all[resto] / p_all[resto].sum()


def muestra_ajuste(seed, n=N_FIT):
    idx = np.random.default_rng(seed).choice(resto, size=n, replace=False, p=p_resto)
    return idx, idx[:N_SUB]


idx_fit, idx_sub = muestra_ajuste(RANDOM_STATE)
print(f"eval={len(idx_eval):,}  fit={len(idx_fit):,}  sub={len(idx_sub):,}")

### 4.3 Espacio de referencia común

El silhouette de cada configuración se calcula en su propio espacio, por lo que no es
comparable entre conjuntos de variables. Además, variables discretas como sexo inflan el
silhouette sin aportar perfiles de participación. Por eso se calcula también un
**silhouette de referencia** en un espacio común: las 6 tasas federales y PJ 2025 estandarizadas.

In [ ]:
REF = ELECCIONES + ["P_PJ2025"]
ref_scaler = StandardScaler().fit(celdas.loc[idx_fit, REF])
X_ref = ref_scaler.transform(celdas.loc[idx_eval, REF])

prom_eval = celdas.loc[idx_eval, "promedio"].to_numpy()
edad_eval = celdas.loc[idx_eval, "GRUPO_EDAD"].to_numpy()
sexo_eval = celdas.loc[idx_eval, "SEXO"].to_numpy()
print(f"Espacio de referencia: {X_ref.shape}")

### 4.4 Preprocesamiento

Estándar, robusto (mediana e IQR, menos sensible a extremos) y estándar + PCA
(90% de varianza; solo con más de 3 variables).

In [ ]:
PREPROCESOS = ["estándar", "robusto", "estandar_pca"]


def preprocesador(nombre):
    if nombre == "estándar":
        return StandardScaler()
    if nombre == "robusto":
        return RobustScaler()
    return make_pipeline(StandardScaler(), PCA(n_components=0.9, random_state=RANDOM_STATE))


def preparar(conjunto, prep, idx_ajuste):
    X = celdas[CONJUNTOS[conjunto]].to_numpy(dtype=float)
    tr = preprocesador(prep).fit(X[idx_ajuste])
    return tr, X


COMBOS = [
    (c, pre) for c, cols in CONJUNTOS.items() for pre in PREPROCESOS
    if not (pre == "estandar_pca" and len(cols) <= 3)
]
print(f"Combinaciones variables x preprocesamiento: {len(COMBOS)}")

### 4.5 Algoritmos e hiperparametros

| Algoritmo | Hiperparametros |
|---|---|
| KMeans | k |
| BisectingKMeans | k |
| Birch | k |
| GMM | k, tipo de covarianza (full, diag, tied, spherical) |
| Aglomerativo | k, enlace (ward, average, complete) |
| Espectral | k (embedding por vecinos cercanos + KMeans) |
| HDBSCAN | tamaño mínimo de cluster, min_samples, método de selección |

DBSCAN se omite porque HDBSCAN lo generaliza sin fijar `eps`. MeanShift y
AffinityPropagation se omiten por costo cuadratico y porque no controlan k.

In [ ]:
def malla_algoritmos():
    malla = []
    for k in K_RANGE:
        malla += [("kmeans", {"k": k}), ("bisecting", {"k": k}),
                  ("birch", {"k": k}), ("espectral", {"k": k})]
        malla += [("gmm", {"k": k, "cov": cov}) for cov in ["full", "diag", "tied", "spherical"]]
        malla += [("aglomerativo", {"k": k, "link": l}) for l in ["ward", "average", "complete"]]
    for mcs in [80, 200, 400]:
        for ms in [5, 25]:
            for sel in ["eom", "leaf"]:
                malla.append(("hdbscan", {"mcs": mcs, "ms": ms, "sel": sel}))
    return malla


MALLA = malla_algoritmos()
print(f"Configuraciones por combinacion: {len(MALLA)}")
print(f"Total de configuraciones: {len(MALLA) * len(COMBOS):,}")

Cada algoritmo devuelve un objeto con `predict`, para etiquetar cualquier conjunto de filas.
Los algoritmos sin `predict` (aglomerativo, espectral, HDBSCAN) se envuelven en un KNN.
El `cache` reutiliza el árbol de Birch, el árbol jerárquico y el embedding espectral entre valores de k.

In [ ]:
def ajustar(algo, prm, Z_fit, Z_sub, seed, cache, w=None, n_init=None):
    extra = {}
    if algo == "kmeans":
        return KMeans(prm["k"], n_init=n_init or 4, random_state=seed).fit(Z_fit, sample_weight=w), extra
    if algo == "bisecting":
        m = BisectingKMeans(prm["k"], n_init=n_init or 2, random_state=seed)
        return m.fit(Z_fit, sample_weight=w), extra
    if algo == "birch":
        if "birch" not in cache:
            cache["birch"] = Birch(n_clusters=None, threshold=0.5).fit(Z_sub)
        m = copy.deepcopy(cache["birch"])
        m.set_params(n_clusters=prm["k"])
        return m.partial_fit(), extra
    if algo == "gmm":
        m = GaussianMixture(prm["k"], covariance_type=prm["cov"], n_init=n_init or 2,
                            random_state=seed).fit(Z_fit)
        extra["bic"] = m.bic(Z_fit)
        return m, extra

    if algo == "aglomerativo":
        if prm["link"] not in cache:
            cache[prm["link"]] = linkage(Z_sub, method=prm["link"])
        lab = fcluster(cache[prm["link"]], prm["k"], criterion="maxclust") - 1
    elif algo == "espectral":
        if "emb" not in cache:
            cache["emb"] = SpectralEmbedding(
                n_components=max(K_RANGE), affinity="nearest_neighbors",
                n_neighbors=15, random_state=seed,
            ).fit_transform(Z_sub)
        lab = KMeans(prm["k"], n_init=4, random_state=seed).fit_predict(cache["emb"][:, :prm["k"]])
    else:
        lab = HDBSCAN(min_cluster_size=prm["mcs"], min_samples=prm["ms"],
                      cluster_selection_method=prm["sel"]).fit_predict(Z_sub)
    return KNeighborsClassifier(n_neighbors=5).fit(Z_sub, lab), extra

### 4.6 Métricas

Todas sobre la muestra eval (ponderada por construcción). En HDBSCAN el ruido (etiqueta -1)
se excluye de las métricas y se reporta aparte.

- **sil, db, ch:** silhouette, Davies-Bouldin y Calinski-Harabasz en el espacio propio
  (los silhouette se calculan sobre una submuestra fija de 5k por costo).
- **sil_ref:** silhouette en el espacio de referencia común.
- **r2_prom:** proporción de la varianza del promedio de participación explicada por los perfiles.
- **v_edad, v_sexo:** V de Cramer entre perfil y grupo de edad o sexo (informativas, no se usan para elegir).
- **min_pct, pct_ruido, k_eff:** tamaño del perfil más chico, ruido y número efectivo de perfiles.

In [ ]:
def cramers_v(a, b):
    tabla = pd.crosstab(a, b).to_numpy()
    if min(tabla.shape) < 2:
        return 0.0
    chi2 = chi2_contingency(tabla, correction=False)[0]
    return float(np.sqrt(chi2 / (tabla.sum() * (min(tabla.shape) - 1))))


def eta2(y, labels):
    medias = pd.Series(y).groupby(labels).transform("mean").to_numpy()
    return float(((medias - y.mean()) ** 2).sum() / ((y - y.mean()) ** 2).sum())


EN_SIL = np.zeros(N_EVAL, dtype=bool)
EN_SIL[np.random.default_rng(RANDOM_STATE).permutation(N_EVAL)[:N_SIL]] = True


def silhouette_seguro(Z, lab):
    return silhouette_score(Z, lab) if len(np.unique(lab)) > 1 else np.nan


def evaluar(labels, Z_eval):
    mask = labels >= 0
    m_sil = mask & EN_SIL
    lab = labels[mask]
    tamanos = pd.Series(lab).value_counts() / len(labels) * 100
    res = {"k_eff": len(tamanos), "pct_ruido": (1 - mask.mean()) * 100,
           "min_pct": tamanos.min() if len(tamanos) else 0.0}
    if len(tamanos) < 2:
        return res
    res.update({
        "sil": silhouette_seguro(Z_eval[m_sil], labels[m_sil]),
        "db": davies_bouldin_score(Z_eval[mask], lab),
        "ch": calinski_harabasz_score(Z_eval[mask], lab),
        "sil_ref": silhouette_seguro(X_ref[m_sil], labels[m_sil]),
        "r2_prom": eta2(prom_eval[mask], lab),
        "v_edad": cramers_v(lab, edad_eval[mask]),
        "v_sexo": cramers_v(lab, sexo_eval[mask]),
    })
    return res

## 5. Búsqueda en malla (etapa 1)

Se recorre cada combinacion de variables y preprocesamiento con todas las configuraciones
de algoritmo. Se guardan métricas, tiempo y las etiquetas de la muestra eval.

In [ ]:
CONFIGS = {}
ETIQUETAS = {}
registros = []
t_inicio = time.perf_counter()

for conjunto, prep in tqdm(COMBOS, desc="combinaciones"):
    tr, X = preparar(conjunto, prep, idx_fit)
    Z_fit, Z_eval = tr.transform(X[idx_fit]), tr.transform(X[idx_eval])
    Z_sub = Z_fit[:N_SUB]
    cache = {}
    for algo, prm in MALLA:
        cid = len(CONFIGS)
        CONFIGS[cid] = (conjunto, prep, algo, prm)
        t0 = time.perf_counter()
        modelo, extra = ajustar(algo, prm, Z_fit, Z_sub, RANDOM_STATE, cache)
        labels = modelo.predict(Z_eval)
        ETIQUETAS[cid] = labels
        registros.append({
            "id": cid, "conjunto": conjunto, "prep": prep, "algoritmo": algo,
            "params": ", ".join(f"{k}={v}" for k, v in prm.items()), "k": prm.get("k"),
            "dim": Z_fit.shape[1], **evaluar(labels, Z_eval), **extra,
            "seg": time.perf_counter() - t0,
        })

res = pd.DataFrame(registros)
print(f"{len(res):,} configuraciones en {(time.perf_counter() - t_inicio) / 60:.1f} min")

### 5.1 Filtro de validez y puntaje

Una configuración es valida si produce 4-6 perfiles, ninguno menor al 5% de la población,
y menos de 10% de ruido. Entre las validas, el puntaje es el promedio de los rangos
percentiles de `sil`, `sil_ref` y `r2_prom`: separación propia, separación en conducta
y capacidad de explicar el nivel de participación.

In [ ]:
def motivo(r):
    if r.k_eff not in K_VALIDO:
        return "k fuera de 4-6"
    if r.pct_ruido > MAX_PCT_RUIDO:
        return "ruido > 10%"
    if r.min_pct < MIN_PCT_PERFIL:
        return "perfil < 5%"
    return "valida"


res["motivo"] = res.apply(motivo, axis=1)
validas = res["motivo"] == "valida"
res["score"] = np.nan
res.loc[validas, "score"] = (
    res.loc[validas, ["sil", "sil_ref", "r2_prom"]].rank(pct=True).mean(axis=1)
)
print(f"Validas: {validas.sum():,} de {len(res):,}")

### 5.2 Descartes por algoritmo

Configuraciones con k = 3, 7 u 8 se incluyen solo para ver la tendencia; siempre caen en
"k fuera de 4-6".

In [ ]:
pd.crosstab(res["algoritmo"], res["motivo"], margins=True)

### 5.3 Mejor configuración por algoritmo y conjunto de variables

Puntaje máximo entre configuraciones validas. Celdas vacias: ninguna configuración valida.

In [ ]:
tabla = res.pivot_table(index="conjunto", columns="algoritmo", values="score", aggfunc="max")

fig, ax = plt.subplots(figsize=(11, 5.5))
im = ax.imshow(tabla.to_numpy(), cmap="viridis", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(tabla.shape[1]), tabla.columns, rotation=30, ha="right")
ax.set_yticks(range(tabla.shape[0]), tabla.index)
for i in range(tabla.shape[0]):
    for j in range(tabla.shape[1]):
        v = tabla.iat[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="white" if v < 0.6 else "black", fontsize=8)
fig.colorbar(im, ax=ax, label="Puntaje máximo")
ax.set_title("Mejor puntaje valido por conjunto de variables y algoritmo")
plt.tight_layout()
plt.show()

### 5.4 Separación en conducta segun k

Silhouette de referencia máximo por algoritmo y k (todas las combinaciones de variables,
sin filtro de validez). Muestra si la estructura mejora o empeora al aumentar k.

In [ ]:
curvas = res.dropna(subset=["k"]).groupby(["algoritmo", "k"])["sil_ref"].max().unstack(0)

fig, ax = plt.subplots(figsize=(9, 4.5))
curvas.plot(ax=ax, marker="o")
ax.axvspan(K_VALIDO.start, K_VALIDO.stop - 1, alpha=0.1, color="blue")
ax.set_xlabel("k")
ax.set_ylabel("sil_ref máximo")
ax.set_title("Silhouette en espacio de referencia vs k (zona azul: 4-6)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 5.5 Efecto de agregar demografia

Comparación de métricas promedio entre configuraciones validas, por grupo de variables.
Si la demografia sube `sil` pero baja `sil_ref` y `r2_prom`, los perfiles se están
separando por edad o sexo y no por conducta.

In [ ]:
grupo = res["conjunto"].str[0].map(
    {"A": "conducta", "B": "conducta", "C": "conducta", "D": "conducta",
     "E": "conducta + demo", "F": "conducta + demo", "G": "conducta + demo",
     "H": "conducta + demo", "I": "solo demo"}
)
res["grupo"] = grupo
(res[validas].groupby("grupo")[["sil", "sil_ref", "r2_prom", "v_edad", "v_sexo"]]
 .mean().round(3))

### 5.6 Top 20 configuraciones validas

In [ ]:
COLS_TABLA = ["id", "conjunto", "prep", "algoritmo", "params", "k_eff", "min_pct",
              "sil", "sil_ref", "r2_prom", "db", "v_edad", "v_sexo", "score"]
res[validas].nlargest(20, "score")[COLS_TABLA].round(3)

## 6. Estabilidad (etapa 2)

Candidatas: las 15 mejores por puntaje más la mejor de cada algoritmo. Cada una se
reajusta con 5 muestras de ajuste nuevas (y semillas distintas) y se compara contra sus
etiquetas originales en la muestra eval con el índice de Rand ajustado (ARI).
Un ARI cercano a 1 indica que los perfiles no dependen de la muestra.

In [ ]:
mejores_algo = res[validas].loc[res[validas].groupby("algoritmo")["score"].idxmax()]
candidatas = pd.concat([res[validas].nlargest(N_CANDIDATOS, "score"), mejores_algo])
candidatas = candidatas.drop_duplicates("id").sort_values("score", ascending=False)
print(f"Candidatas: {len(candidatas)}")

In [ ]:
estabilidad = []
for cid in tqdm(candidatas["id"], desc="bootstrap"):
    conjunto, prep, algo, prm = CONFIGS[cid]
    aris = []
    for b in range(N_BOOT):
        seed = 1000 + b
        idx_f, _ = muestra_ajuste(seed)
        tr, X = preparar(conjunto, prep, idx_f)
        Z_f = tr.transform(X[idx_f])
        modelo, _ = ajustar(algo, prm, Z_f, Z_f[:N_SUB], seed, {})
        aris.append(adjusted_rand_score(ETIQUETAS[cid], modelo.predict(tr.transform(X[idx_eval]))))
    estabilidad.append({"id": cid, "ari_media": np.mean(aris), "ari_min": np.min(aris)})

candidatas = candidatas.merge(pd.DataFrame(estabilidad), on="id")
candidatas[COLS_TABLA + ["ari_media", "ari_min"]].round(3)

### 6.1 Finalistas

Se descartan las candidatas con ARI medio menor a 0.8. De las estables, pasan las 3 de
mayor puntaje.

In [ ]:
estables = candidatas[candidatas["ari_media"] >= ARI_ESTABLE]
if len(estables) == 0:
    print("Ninguna candidata supera el umbral de ARI; se usan las más estables.")
    estables = candidatas.nlargest(N_FINALISTAS, "ari_media")
finalistas = estables.nlargest(N_FINALISTAS, "score")
print(f"Estables: {len(estables)} de {len(candidatas)}")
finalistas[COLS_TABLA + ["ari_media"]].round(3)

## 7. Afinacion y modelo final (etapa 3)

Las finalistas se reajustan con más datos y más inicializaciones:

- KMeans y BisectingKMeans: todas las celdas con `sample_weight` y `n_init=20`.
- GMM: muestra ponderada de 200k con `n_init=5`.
- Birch, aglomerativo, espectral y HDBSCAN: muestra sub y asignación a todas las celdas.

Se agrega como referencia el modelo del notebook 04 (A_base, estándar, KMeans k=4).

In [ ]:
def ajustar_final(conjunto, prep, algo, prm):
    tr, X = preparar(conjunto, prep, idx_fit)
    Z_all = tr.transform(X)
    if algo in ("kmeans", "bisecting"):
        modelo, _ = ajustar(algo, prm, Z_all, None, RANDOM_STATE, {},
                            w=celdas["peso"].to_numpy(), n_init=20)
    elif algo == "gmm":
        idx_big, _ = muestra_ajuste(RANDOM_STATE, n=N_FINAL)
        modelo, _ = ajustar(algo, prm, Z_all[idx_big], None, RANDOM_STATE, {}, n_init=5)
    else:
        modelo, _ = ajustar(algo, prm, None, Z_all[idx_sub], RANDOM_STATE, {})
    return modelo.predict(Z_all), Z_all


modelos_finales = {"base_04": ("A_base", "estándar", "kmeans", {"k": 4})}
for i, r in enumerate(finalistas.itertuples(), start=1):
    modelos_finales[f"candidato_{i}"] = CONFIGS[r.id]

etiquetas_finales, metricas_finales = {}, []
for nombre, (conjunto, prep, algo, prm) in tqdm(modelos_finales.items(), desc="modelos finales"):
    labels, Z_all = ajustar_final(conjunto, prep, algo, prm)
    etiquetas_finales[nombre] = labels
    metricas_finales.append({
        "modelo": nombre, "conjunto": conjunto, "prep": prep, "algoritmo": algo,
        "params": ", ".join(f"{k}={v}" for k, v in prm.items()),
        **evaluar(labels[idx_eval], Z_all[idx_eval]),
    })

metricas_finales = pd.DataFrame(metricas_finales)
metricas_finales.round(3)

### 7.1 Coincidencia entre modelos

ARI entre las particiones finales, medido sobre la muestra eval. Valores bajos frente a
`base_04` indican que el candidato propone perfiles distintos a los actuales.

In [ ]:
nombres = list(etiquetas_finales)
ari = pd.DataFrame(
    [[adjusted_rand_score(etiquetas_finales[a][idx_eval], etiquetas_finales[b][idx_eval])
      for b in nombres] for a in nombres],
    index=nombres, columns=nombres,
)
ari.round(3)

### 7.2 Perfiles de cada modelo

Resumen ponderado por peso. Los perfiles se ordenan por promedio de participación.
La caracterización detallada y los nombres se hacen después, una vez elegido el modelo.

In [ ]:
def resumen_perfiles(labels):
    df = celdas.assign(perfil=labels)
    df = df[df["perfil"] >= 0]
    filas = []
    for p, dp in df.groupby("perfil"):
        w = dp["peso"]
        edad = dp.groupby("GRUPO_EDAD")["peso"].sum()
        filas.append({
            "pct_pob": w.sum() / df["peso"].sum() * 100,
            "promedio": np.average(dp["promedio"], weights=w),
            "brecha_pres": np.average(dp["brecha_pres"], weights=w),
            "tendencia": np.average(dp["tendencia"], weights=w),
            "P_PJ2025": np.average(dp["P_PJ2025"], weights=w),
            "pct_mujer": np.average(dp["MUJER"], weights=w) * 100,
            "pct_rural_mixta": np.average(dp["TIPOSEC_ORD"] > 0, weights=w) * 100,
            "edad_dominante": edad.idxmax(),
        })
    return pd.DataFrame(filas).sort_values("promedio").reset_index(drop=True).round(1)


for nombre, labels in etiquetas_finales.items():
    conjunto, prep, algo, prm = modelos_finales[nombre]
    print(f"\n{nombre}: {conjunto} | {prep} | {algo} {prm}")
    print(resumen_perfiles(labels).to_string())

## 8. Exportacion

No se sobrescribe nada del notebook 04. Se guardan los resultados de la malla,
la estabilidad de las candidatas y las etiquetas de los modelos finales por celda.

In [ ]:
res.drop(columns=["grupo"]).to_csv(DIR_PREPROCESSED / "experimentos_clustering.csv", index=False)
candidatas.to_csv(DIR_PREPROCESSED / "experimentos_estabilidad.csv", index=False)
metricas_finales.to_csv(DIR_PREPROCESSED / "experimentos_finalistas.csv", index=False)

salida = celdas[["EDOCVE", "SECCIÓN", "SEXO", "GRUPO_EDAD", "peso"]].copy()
for nombre, labels in etiquetas_finales.items():
    salida[nombre] = labels.astype("int8")
salida.to_parquet(DIR_PREPROCESSED / "celdas_perfiles_candidatos.parquet", index=False)

for f in ["experimentos_clustering.csv", "experimentos_estabilidad.csv",
          "experimentos_finalistas.csv", "celdas_perfiles_candidatos.parquet"]:
    print(f"{f}: {(DIR_PREPROCESSED / f).stat().st_size / 1024:.0f} KB")